## **Install Libraries**

In [ ]:
!pip install --upgrade langchain
!pip install --upgrade langchain-community
!pip install --upgrade langchain-openai
!pip install --upgrade langchain-text-splitters
!pip install --upgrade langchain-groq
!pip install --upgrade langchain-huggingface
!pip install chromadb
!pip install pypdf

# **Import libraries**

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from google.colab import files
import os

# **GROQ API Key**

In [ ]:
os.environ["GROQ_API_KEY"] = "API_KEY"

In [ ]:
os.environ["OPENAI_API_KEY"] = "API_KEY" # Replace with your OpenAI API Key

# **LLM Instance Creation**

In [ ]:
os.environ["GROQ_API_KEY"] = "API_KEY"
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.3-70b-versatile",   # Best balance of quality + speed
    temperature=0)

# **Read the document**

In [ ]:
# Importing the training data
# Prompt user to upload file
uploaded = files.upload()

# Get uploaded filename
file_name = list(uploaded.keys())[0]

Saving CBUAE_EN_5996_VER1.pdf to CBUAE_EN_5996_VER1.pdf


In [ ]:
with open(file_name, "wb") as f:
    f.write(uploaded[file_name])

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader(file_name)
documents = loader.load()

In [ ]:
len(documents)

44

# **Split the documents, Chunk it**

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

# **Create Embeddings**

In [ ]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# **Strore in Vector DB or Chroma DB**

In [ ]:
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="policy_db"
)

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
vectorstore.persist()

/tmp/ipykernel_1522/398866168.py:1: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  vectorstore.persist()


# **Shoot the question and get the answer**

In [ ]:
while True:
    query = input("\nEnter your question (or type 'exit' to quit): ")

    if query.lower() == "exit":
        print("Goodbye!")
        break

    results = vectorstore.similarity_search(query, k=3)

    context = "\n\n".join(doc.page_content for doc in results)

    prompt = f"""
    Answer the question using only the context below.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    print("\nAnswer:")
    print(response.content)


Answer:
The provided context does not explicitly define "Credit Risk Governance." However, based on the information given, it can be inferred that Credit Risk Governance refers to the framework, policies, processes, and responsibilities established to identify, measure, evaluate, monitor, report, and control or mitigate Credit Risk within a financial institution, ensuring that these activities are consistent with the institution's Risk Appetite, strategy, and tolerances. 

This includes the oversight role of the Board, the responsibilities of the Credit Risk Management Function, and the duties of Senior Management in managing Credit Risk, all of which contribute to a comprehensive governance structure for Credit Risk.

Answer:
According to the context, an Obligor is considered Past Due (and potentially in default) after being more than 90 days past due on any material credit obligation to the LFI.

Answer:
According to the context, 3 or more deferrals of consecutive instalments trigge